# Chunking for Document Types
### HTML & Code — Format-Aware Text Splitting Strategies

Two small inline documents instead of a PDF corpus: a mini HTML product page for Part A, and a mini Python module for Part B — small enough to read in full, so the effect of each chunking strategy is visible chunk-by-chunk. Each part ends by embedding its own chunks and running a real retrieve-then-generate pass, so bad chunking would show up as a bad answer.

## Step 1: Build the pipeline

In [1]:
!pip install langchain langchain-community langchain-ollama langchain-text-splitters faiss-cpu beautifulsoup4 -q


[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from bs4 import BeautifulSoup
from langchain_text_splitters import HTMLHeaderTextSplitter, RecursiveCharacterTextSplitter, Language
from langchain_community.vectorstores import FAISS
from langchain_ollama import OllamaEmbeddings, ChatOllama

embeddings = OllamaEmbeddings(model="nomic-embed-text:latest")
llm = ChatOllama(model="llama3.2:3b", temperature=0)

C:\Users\shiva\.pyenv\pyenv-win\versions\3.12.10\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


C:\Users\shiva\AppData\Local\Temp\ipykernel_25416\1021927784.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


## Part A: Chunking HTML Documents

## Step 2: The raw HTML page
Boilerplate (`nav`, `footer`, `script`, `style`) is mixed in with the real content, exactly like a scraped documentation page.

In [3]:
sample_html = """
<html>
<head><style>body { font-family: sans-serif; }</style></head>
<body>
<nav><a href="/">Home</a> <a href="/docs">Docs</a> <a href="/pricing">Pricing</a></nav>
<script>trackPageView();</script>

<h1>Vector Database Guide</h1>
<p>This guide covers how to choose and configure a vector database for production RAG systems.</p>

<h2>Installation</h2>
<p>Install the client library with pip and start a local instance using Docker. Most vector databases expose a REST or gRPC API on port 6333 or 8080.</p>

<h2>Indexing Strategies</h2>
<p>The index determines how vectors are organized for fast approximate nearest-neighbor search.</p>

<h3>HNSW</h3>
<p>Hierarchical Navigable Small World graphs give the best query speed and recall for most workloads, at the cost of higher memory usage and slower build time.</p>

<h3>IVF</h3>
<p>Inverted File indexes cluster vectors into buckets first, then search only the nearest buckets. They use less memory than HNSW but need a training step on representative data.</p>

<h2>Querying</h2>
<p>Queries specify a vector, a top-k count, and optional metadata filters. Set ef_search higher for HNSW if recall is too low.</p>

<footer>&copy; 2026 Nunnari Academy. All rights reserved.</footer>
</body>
</html>
"""
print(sample_html)


<html>
<head><style>body { font-family: sans-serif; }</style></head>
<body>
<nav><a href="/">Home</a> <a href="/docs">Docs</a> <a href="/pricing">Pricing</a></nav>
<script>trackPageView();</script>

<h1>Vector Database Guide</h1>
<p>This guide covers how to choose and configure a vector database for production RAG systems.</p>

<h2>Installation</h2>
<p>Install the client library with pip and start a local instance using Docker. Most vector databases expose a REST or gRPC API on port 6333 or 8080.</p>

<h2>Indexing Strategies</h2>
<p>The index determines how vectors are organized for fast approximate nearest-neighbor search.</p>

<h3>HNSW</h3>
<p>Hierarchical Navigable Small World graphs give the best query speed and recall for most workloads, at the cost of higher memory usage and slower build time.</p>

<h3>IVF</h3>
<p>Inverted File indexes cluster vectors into buckets first, then search only the nearest buckets. They use less memory than HNSW but need a training step on representati

## Step 3: Strip boilerplate before splitting
`HTMLHeaderTextSplitter` would otherwise turn the nav links and the analytics snippet into meaningless chunks with no header context.

In [4]:
soup = BeautifulSoup(sample_html, "html.parser")
for tag in soup.find_all(["nav", "footer", "script", "style"]):
    tag.decompose()

cleaned_html = str(soup)
print(cleaned_html)


<html>
<head></head>
<body>


<h1>Vector Database Guide</h1>
<p>This guide covers how to choose and configure a vector database for production RAG systems.</p>
<h2>Installation</h2>
<p>Install the client library with pip and start a local instance using Docker. Most vector databases expose a REST or gRPC API on port 6333 or 8080.</p>
<h2>Indexing Strategies</h2>
<p>The index determines how vectors are organized for fast approximate nearest-neighbor search.</p>
<h3>HNSW</h3>
<p>Hierarchical Navigable Small World graphs give the best query speed and recall for most workloads, at the cost of higher memory usage and slower build time.</p>
<h3>IVF</h3>
<p>Inverted File indexes cluster vectors into buckets first, then search only the nearest buckets. They use less memory than HNSW but need a training step on representative data.</p>
<h2>Querying</h2>
<p>Queries specify a vector, a top-k count, and optional metadata filters. Set ef_search higher for HNSW if recall is too low.</p>

</body>
</

## Step 4: Split by DOM header hierarchy
Each chunk carries its `h1`/`h2`/`h3` ancestry as metadata — that heading path becomes retrieval context, not just the chunk text.

In [5]:
headers_to_split_on = [
    ("h1", "Header 1"),
    ("h2", "Header 2"),
    ("h3", "Header 3"),
]
html_splitter = HTMLHeaderTextSplitter(headers_to_split_on=headers_to_split_on)
html_chunks = html_splitter.split_text(cleaned_html)

print(f"{len(html_chunks)} chunks\n")
for chunk in html_chunks:
    print(chunk.metadata, "->", chunk.page_content[:80])

12 chunks

{'Header 1': 'Vector Database Guide'} -> Vector Database Guide
{'Header 1': 'Vector Database Guide'} -> This guide covers how to choose and configure a vector database for production R
{'Header 1': 'Vector Database Guide', 'Header 2': 'Installation'} -> Installation
{'Header 1': 'Vector Database Guide', 'Header 2': 'Installation'} -> Install the client library with pip and start a local instance using Docker. Mos
{'Header 1': 'Vector Database Guide', 'Header 2': 'Indexing Strategies'} -> Indexing Strategies
{'Header 1': 'Vector Database Guide', 'Header 2': 'Indexing Strategies'} -> The index determines how vectors are organized for fast approximate nearest-neig
{'Header 1': 'Vector Database Guide', 'Header 2': 'Indexing Strategies', 'Header 3': 'HNSW'} -> HNSW
{'Header 1': 'Vector Database Guide', 'Header 2': 'Indexing Strategies', 'Header 3': 'HNSW'} -> Hierarchical Navigable Small World graphs give the best query speed and recall f
{'Header 1': 'Vector Database Guide', 'He

# HEader1
## Header2
### Header 3

## Step 5: Embed the chunks and retrieve

In [6]:
html_vector_store = FAISS.from_documents(html_chunks, embeddings)

query = "Which indexing strategy uses less memory but needs a training step?"
retrieved = html_vector_store.similarity_search(query, k=2)

print("Retrieved chunks:")
for doc in retrieved:
    print(doc.metadata, "->", doc.page_content)

Retrieved chunks:
{'Header 1': 'Vector Database Guide', 'Header 2': 'Indexing Strategies', 'Header 3': 'IVF'} -> Inverted File indexes cluster vectors into buckets first, then search only the nearest buckets. They use less memory than HNSW but need a training step on representative data.
{'Header 1': 'Vector Database Guide', 'Header 2': 'Indexing Strategies', 'Header 3': 'HNSW'} -> Hierarchical Navigable Small World graphs give the best query speed and recall for most workloads, at the cost of higher memory usage and slower build time.


## Step 6: Generate the answer from the retrieved chunk

In [7]:
context = "\n\n".join(doc.page_content for doc in retrieved)
prompt = f"""Answer the question based only on the following context:

{context}

Question: {query}
Answer:"""

print(llm.invoke(prompt).content)

Inverted File indexes.


## Part B: Chunking Code Files

## Step 7: The raw Python source
One module: two standalone functions and a class with three methods, written without the usual blank lines between them (as if it came through a minifier or a copy-paste from a diff) — compact enough that the two splitters below are forced to make a real choice about where to cut.

In [8]:
sample_code = '''import math
def normalize(vector):
    """Scale a vector to unit length."""
    magnitude = math.sqrt(sum(x * x for x in vector))
    if magnitude == 0:
        return vector
    return [x / magnitude for x in vector]
def cosine_similarity(vec_a, vec_b):
    """Return the cosine similarity between two equal-length vectors."""
    dot = sum(a * b for a, b in zip(vec_a, vec_b))
    norm_a = math.sqrt(sum(a * a for a in vec_a))
    norm_b = math.sqrt(sum(b * b for b in vec_b))
    if norm_a == 0 or norm_b == 0:
        return 0.0
    return dot / (norm_a * norm_b)
class VectorStore:
    """A minimal in-memory vector store for nearest-neighbor search."""
    def __init__(self):
        self.vectors = []
        self.payloads = []
    def add(self, vector, payload):
        """Store a vector alongside its associated payload."""
        self.vectors.append(normalize(vector))
        self.payloads.append(payload)
    def search(self, query_vector, top_k=3):
        """Return the top_k payloads ranked by cosine similarity to the query."""
        query_vector = normalize(query_vector)
        scored = [
            (cosine_similarity(query_vector, v), p)
            for v, p in zip(self.vectors, self.payloads)
        ]
        scored.sort(key=lambda pair: pair[0], reverse=True)
        return scored[:top_k]
    def delete(self, payload):
        """Remove the first stored vector matching the given payload."""
        if payload in self.payloads:
            index = self.payloads.index(payload)
            del self.vectors[index]
            del self.payloads[index]
'''
print(sample_code)

import math
def normalize(vector):
    """Scale a vector to unit length."""
    magnitude = math.sqrt(sum(x * x for x in vector))
    if magnitude == 0:
        return vector
    return [x / magnitude for x in vector]
def cosine_similarity(vec_a, vec_b):
    """Return the cosine similarity between two equal-length vectors."""
    dot = sum(a * b for a, b in zip(vec_a, vec_b))
    norm_a = math.sqrt(sum(a * a for a in vec_a))
    norm_b = math.sqrt(sum(b * b for b in vec_b))
    if norm_a == 0 or norm_b == 0:
        return 0.0
    return dot / (norm_a * norm_b)
class VectorStore:
    """A minimal in-memory vector store for nearest-neighbor search."""
    def __init__(self):
        self.vectors = []
        self.payloads = []
    def add(self, vector, payload):
        """Store a vector alongside its associated payload."""
        self.vectors.append(normalize(vector))
        self.payloads.append(payload)
    def search(self, query_vector, top_k=3):
        """Return the top_k payload

## Step 8: Baseline — naive character splitting
A plain `RecursiveCharacterTextSplitter` doesn't know what a function or class is — it just counts characters, so it can cut a signature away from its body.

In [9]:
naive_splitter = RecursiveCharacterTextSplitter(chunk_size=260, chunk_overlap=0)
naive_chunks = naive_splitter.split_text(sample_code)

print(f"{len(naive_chunks)} naive chunks\n")
for i, chunk in enumerate(naive_chunks):
    print(f"--- chunk {i} ---")
    print(chunk)
    print()

7 naive chunks

--- chunk 0 ---
import math
def normalize(vector):
    """Scale a vector to unit length."""
    magnitude = math.sqrt(sum(x * x for x in vector))
    if magnitude == 0:
        return vector
    return [x / magnitude for x in vector]
def cosine_similarity(vec_a, vec_b):

--- chunk 1 ---
"""Return the cosine similarity between two equal-length vectors."""
    dot = sum(a * b for a, b in zip(vec_a, vec_b))
    norm_a = math.sqrt(sum(a * a for a in vec_a))
    norm_b = math.sqrt(sum(b * b for b in vec_b))
    if norm_a == 0 or norm_b == 0:

--- chunk 2 ---
return 0.0
    return dot / (norm_a * norm_b)
class VectorStore:
    """A minimal in-memory vector store for nearest-neighbor search."""
    def __init__(self):
        self.vectors = []
        self.payloads = []
    def add(self, vector, payload):

--- chunk 3 ---
"""Store a vector alongside its associated payload."""
        self.vectors.append(normalize(vector))
        self.payloads.append(payload)
    def search(se

Chunk 0 ends on the bare line `def cosine_similarity(vec_a, vec_b):` — the signature is cut away from its own docstring and body, which only appear at the start of chunk 1. A retriever that returns chunk 0 alone hands the LLM a function name with no implementation.

## Step 9: Language-aware splitting
`RecursiveCharacterTextSplitter.from_language(Language.PYTHON, ...)` uses a separator priority of class → def → block → line, so it prefers to end a chunk *before* the next `def`/`class`, not partway through one.

In [10]:
python_splitter = RecursiveCharacterTextSplitter.from_language(
    language=Language.PYTHON, chunk_size=260, chunk_overlap=0
)
code_chunks = python_splitter.create_documents([sample_code])

print(f"{len(code_chunks)} language-aware chunks\n")
for i, doc in enumerate(code_chunks):
    print(f"--- chunk {i} ---")
    print(doc.page_content)
    print()

8 language-aware chunks

--- chunk 0 ---
import math
def normalize(vector):
    """Scale a vector to unit length."""
    magnitude = math.sqrt(sum(x * x for x in vector))
    if magnitude == 0:
        return vector
    return [x / magnitude for x in vector]

--- chunk 1 ---
def cosine_similarity(vec_a, vec_b):
    """Return the cosine similarity between two equal-length vectors."""
    dot = sum(a * b for a, b in zip(vec_a, vec_b))
    norm_a = math.sqrt(sum(a * a for a in vec_a))

--- chunk 2 ---
norm_b = math.sqrt(sum(b * b for b in vec_b))
    if norm_a == 0 or norm_b == 0:
        return 0.0
    return dot / (norm_a * norm_b)

--- chunk 3 ---
class VectorStore:
    """A minimal in-memory vector store for nearest-neighbor search."""
    def __init__(self):
        self.vectors = []
        self.payloads = []
    def add(self, vector, payload):

--- chunk 4 ---
"""Store a vector alongside its associated payload."""
        self.vectors.append(normalize(vector))
        self.payloads

Now `normalize` finishes cleanly in chunk 0, and `cosine_similarity` starts fresh in chunk 1 with its signature and docstring intact — the same chunk budget, spent at a better boundary.

## Step 10: Embed the code chunks and retrieve

In [11]:
code_vector_store = FAISS.from_documents(code_chunks, embeddings)

code_query = "How do I compute cosine similarity between two vectors?"
code_retrieved = code_vector_store.similarity_search(code_query, k=2)

print("Retrieved chunks:")
for doc in code_retrieved:
    print(doc.page_content)
    print("---")

Retrieved chunks:
def cosine_similarity(vec_a, vec_b):
    """Return the cosine similarity between two equal-length vectors."""
    dot = sum(a * b for a, b in zip(vec_a, vec_b))
    norm_a = math.sqrt(sum(a * a for a in vec_a))
---
"""Return the top_k payloads ranked by cosine similarity to the query."""
        query_vector = normalize(query_vector)
        scored = [
            (cosine_similarity(query_vector, v), p)
            for v, p in zip(self.vectors, self.payloads)
---


## Step 11: Generate the answer from the retrieved code

In [12]:
code_context = "\n\n".join(doc.page_content for doc in code_retrieved)
code_prompt = f"""Answer the question using only the following code as context:

{code_context}

Question: {code_query}
Answer:"""

print(llm.invoke(code_prompt).content)

You can compute the cosine similarity between two vectors using the `cosine_similarity` function provided:

```python
dot = sum(a * b for a, b in zip(vec_a, vec_b))
norm_a = math.sqrt(sum(a * a for a in vec_a))

return dot / (norm_a * norm_b)
```

However, according to your code snippet, the `cosine_similarity` function is already implemented:

```python
def cosine_similarity(vec_a, vec_b):
    """Return the cosine similarity between two equal-length vectors."""
    dot = sum(a * b for a, b in zip(vec_a, vec_b))
    norm_a = math.sqrt(sum(a * a for a in vec_a))

    return dot / (norm_a * norm_b)
```

Note that `norm_b` is not defined in your code snippet. It should be calculated similarly to `norm_a`.


## Try it yourself
1. Add an `h4` to `headers_to_split_on` and a matching `<h4>` in the HTML — does retrieval still find the right chunk?
2. Drop the code `chunk_size` to 80 — does the language-aware splitter still keep each function's signature and docstring together, or does it start cutting through them too?
3. Swap `Language.PYTHON` for `Language.JS` and pass in a small JavaScript snippet instead.